In [1]:
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain.embeddings import init_embeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from transformers import AutoTokenizer
import chromadb

c:\Users\eparedes\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Autenticación y Configuración de Modelos de OpenAI
with open("api_key.txt") as archivo:
    apikey = archivo.read().strip()
os.environ["OPENAI_API_KEY"] = apikey

from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4o-mini", model_provider="openai")

embeddings = init_embeddings(
    provider="openai",
    model="text-embedding-3-small"
)

In [3]:
# 2. Carga del PDF
pdf_path = "ley31814.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

In [4]:
# 3. Tokenizador e Intercalador Genérico
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)

In [5]:
# 4. Almacén Vectorial Genérico (Chroma DB)
chroma_client = chromadb.EphemeralClient()
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings,
    client=chroma_client,
    collection_name="ley_31814_v1536"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [6]:
# 5. Prompt optimizado para RAG
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente experto que responde preguntas basándose única y exclusivamente en el contexto provisto. Si no sabes la respuesta o no está en el contexto, di que no la conoces."),
    ("user", "Contexto:\n{context}\n\nPregunta: {question}")
])

In [7]:
# 6. Formateador de Documentos
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [8]:
# 7. Cadena LCEL RAG
rag_chain = (
    RunnableParallel({
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    })
    | rag_prompt
    | model
    | StrOutputParser()
)

In [9]:
# 8. Ejecución
query = "¿Cuáles son los puntos clave del documento?"
result = rag_chain.invoke(query)
print(result)

Los puntos clave del documento son:

1. **Objetivo de la Ley**: Promover el uso de la inteligencia artificial en el marco de la transformación digital, priorizando el respeto a los derechos humanos y fomentando el desarrollo económico y social en un entorno seguro.

2. **Interés Nacional**: Destaca la importancia de la promoción del talento digital y el uso de la inteligencia artificial en varios sectores, como salud, educación, justicia, seguridad y economía.

3. **Definiciones**: Incluye definiciones claras de inteligencia artificial, sistemas basados en inteligencia artificial, tecnologías emergentes y algoritmos, estableciendo el contexto técnico de la ley.

4. **Autoridad Nacional**: Establece que la Presidencia del Consejo de Ministros es la entidad responsable de dirigir y supervisar el desarrollo y uso de la inteligencia artificial y tecnologías emergentes.

5. **Informe al Congreso**: Se requiere que la Autoridad Nacional presente un informe anual al Congreso sobre el progreso